In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


In [ ]:
import re
import numpy as np
import pandas as pd
import torch
import evaluate

from datasets import load_dataset
from datasets import Dataset
from datasets import DatasetDict

from transformers import (

AutoTokenizer,
AutoModelForSequenceClassification,
AutoConfig,

TrainingArguments,
Trainer,

EarlyStoppingCallback,
DataCollatorWithPadding

)

from sklearn.model_selection import GroupShuffleSplit

from sklearn.metrics import (

accuracy_score,
precision_recall_fscore_support

)

In [ ]:
SEED = 42

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
dataset = load_dataset("raquiba/Sarcasm_News_Headline")

df = pd.DataFrame(dataset["train"])

df = df[["headline", "is_sarcastic"]]

df.columns = ["text", "label"]

print(df.head())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.json:   0%|          | 0.00/6.03M [00:00<?, ?B/s]

test.json:   0%|          | 0.00/5.59M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/28619 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/26709 [00:00<?, ? examples/s]

                                                text  label
0  thirtysomething scientists unveil doomsday clo...      1
1  dem rep. totally nails why congress is falling...      0
2  eat your veggies: 9 deliciously different recipes      0
3  inclement weather prevents liar from getting t...      1
4  mother comes pretty close to using word 'strea...      1


In [ ]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    return text

In [ ]:
before = len(df)

df = df.drop_duplicates(subset=["text"])

after = len(df)

print("Duplicates removed:", before - after)

Duplicates removed: 116


In [ ]:
df = df[
    (df["text"].str.split().str.len() >= 4)
]

In [ ]:
from sklearn.utils import resample

majority = df[df.label == 0]
minority = df[df.label == 1]

minority_upsampled = resample(

minority,

replace=True,

n_samples=len(majority),

random_state=SEED

)

df = pd.concat([
majority,
minority_upsampled
])

df = df.sample(
frac=1,
random_state=SEED
).reset_index(drop=True)

print(df["label"].value_counts())

label
0    14703
1    14703
Name: count, dtype: int64


In [ ]:
df["group"] = (

df["label"].astype(str)
+ "_"
+ (df.index // 5).astype(str)

)

groups = df["group"]

In [ ]:
gss = GroupShuffleSplit(

n_splits=1,
test_size=0.2,
random_state=SEED

)

train_idx, temp_idx = next(
gss.split(df, groups=groups)
)

train_df = df.iloc[train_idx]
temp_df = df.iloc[temp_idx]

gss2 = GroupShuffleSplit(

n_splits=1,
test_size=0.5,
random_state=SEED

)

val_idx, test_idx = next(

gss2.split(
    temp_df,
    groups=temp_df["group"]
)

)

val_df = temp_df.iloc[val_idx]
test_df = temp_df.iloc[test_idx]

In [ ]:
dataset = DatasetDict({

"train": Dataset.from_pandas(
    train_df[["text", "label"]]
),

"validation": Dataset.from_pandas(
    val_df[["text", "label"]]
),

"test": Dataset.from_pandas(
    test_df[["text", "label"]]
)

})

In [ ]:
model_name = "roberta-large"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

def tokenize_function(example):

    return tokenizer(

        example["text"],

        truncation=True,

        padding="max_length",

        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/23447 [00:00<?, ? examples/s]

Map:   0%|          | 0/3038 [00:00<?, ? examples/s]

Map:   0%|          | 0/2921 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorWithPadding(
tokenizer=tokenizer
)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(

    "roberta-large",

    num_labels=2
)

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    precision, recall, f1, _ = precision_recall_fscore_support(

        labels,

        predictions,

        average="macro"
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {

        "accuracy": accuracy,

        "macro_precision": precision,

        "macro_recall": recall,

        "macro_f1": f1
    }

In [ ]:
training_args = TrainingArguments(

    output_dir="./results",

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="macro_f1",

    greater_is_better=True,

    learning_rate=2e-5,

    weight_decay=0.01,

    warmup_ratio=0.1,

    lr_scheduler_type="cosine",

    max_grad_norm=1.0,

    num_train_epochs=2,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    gradient_accumulation_steps=2,

    fp16=False,

    logging_steps=50,

    save_total_limit=2,

    seed=42,

    report_to="none"
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer = Trainer(

model=model,

args=training_args,

train_dataset=tokenized_dataset["train"],

eval_dataset=tokenized_dataset["validation"],

data_collator=data_collator,

compute_metrics=compute_metrics,

callbacks=[

    EarlyStoppingCallback(
        early_stopping_patience=1
    )
]

)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,0.713628,0.342430,0.933180,0.933863,0.933759,0.933179
2,0.279468,0.287769,0.950296,0.951635,0.949730,0.950189


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=5862, training_loss=0.5734077028673445, metrics={'train_runtime': 4151.6273, 'train_samples_per_second': 11.295, 'train_steps_per_second': 1.412, 'total_flos': 1.0925497337975808e+16, 'train_loss': 0.5734077028673445, 'epoch': 2.0})

In [ ]:
torch.save(
    model.state_dict(),
    "roberta_large_weights.pth"
)

In [ ]:
from google.colab import files

files.download("roberta_large_weights.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls

drive  results	roberta_large_weights.pth  sample_data


In [ ]:
!zip -r roberta_large_small.zip /content/roberta_large_model

	zip warning: name not matched: /content/roberta_large_model

zip error: Nothing to do! (try: zip -r roberta_large_small.zip . -i /content/roberta_large_model)


In [ ]:
SAVE_PATH = "/content/roberta_large_model"

trainer.save_model(SAVE_PATH)

tokenizer.save_pretrained(SAVE_PATH)

print("Model Saved Successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model Saved Successfully!


In [ ]:
!ls /content/roberta_large_model

config.json	   tokenizer_config.json  training_args.bin
model.safetensors  tokenizer.json


In [ ]:
!zip -r roberta_large_small.zip /content/roberta_large_model

  adding: content/roberta_large_model/ (stored 0%)
  adding: content/roberta_large_model/tokenizer.json (deflated 82%)
  adding: content/roberta_large_model/model.safetensors (deflated 9%)
  adding: content/roberta_large_model/training_args.bin (deflated 53%)
  adding: content/roberta_large_model/config.json (deflated 51%)
  adding: content/roberta_large_model/tokenizer_config.json (deflated 50%)


In [ ]:
!ls

drive	 roberta_large_model	  roberta_large_weights.pth
results  roberta_large_small.zip  sample_data


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!cp roberta_large_small.zip "/content/drive/MyDrive/"

In [ ]:
!rm -rf ./results/checkpoint-*